# 03 — Attack, Mask, and Grid Encoding Visualization

Clean in-memory visualizations reusing `coverless_steg` grid helpers without `/test.jpg` side effects.

In [ ]:
from pathlib import Path
import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np

from coverless_steg import (
    H5Entry,
    apply_gaussian_noise,
    apply_jpeg_comp,
    calculate_grid_means,
    compare_neighbors,
    compute_mask_matrices,
)

DATASET_DIR = Path("dataset")
CLEAN_H5 = (
    DATASET_DIR / "SAM_outputs_clean.h5"
    if (DATASET_DIR / "SAM_outputs_clean.h5").exists()
    else DATASET_DIR / "SAM_outputs.h5"
)
NOISE_H5 = DATASET_DIR / "SAM_outputs_noise.h5"

In [ ]:
def visualize_attack(
    original_rgb: np.ndarray,
    attacked_rgb: np.ndarray,
    title: str = "Attacked Image",
    fig_size: tuple[int, int] = (14, 6),
):
    """Display original vs. in-memory attacked RGB images side by side."""
    fig, axes = plt.subplots(1, 2, figsize=fig_size, constrained_layout=True)
    axes[0].imshow(original_rgb)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(attacked_rgb)
    axes[1].set_title(title)
    axes[1].axis("off")
    plt.show()
    return fig

In [ ]:
def _render_mask_overlay(
    segmentations: np.ndarray | None,
    height: int,
    width: int,
    base_rgb: np.ndarray | None = None,
    alpha: float = 0.5,
    seed: int = 42,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    canvas = np.zeros((height, width, 3), dtype=np.uint8) if base_rgb is None else base_rgb.copy()
    if segmentations is None:
        return canvas
    for mask in segmentations:
        seg = mask["segmentation"] if isinstance(mask, dict) else mask
        color = rng.integers(50, 256, size=3, dtype=np.uint8)
        if base_rgb is None:
            canvas[seg] = color
        else:
            canvas[seg] = (alpha * color + (1.0 - alpha) * canvas[seg]).astype(np.uint8)
    return canvas


def visualize_masks(
    entry,
    attacked_entry=None,
    base_rgb: np.ndarray | None = None,
    attacked_rgb: np.ndarray | None = None,
    alpha: float = 0.5,
    fig_size: tuple[int, int] = (14, 6),
):
    """Plot SAM mask overlays for a clean entry and optional attacked entry."""
    seg1 = entry.get("segmentation")
    ov1 = _render_mask_overlay(seg1, int(entry["height"]), int(entry["width"]), base_rgb=base_rgb, alpha=alpha)
    n1 = 0 if seg1 is None else len(seg1)

    cols = 2 if attacked_entry is not None else 1
    fig, axes = plt.subplots(1, cols, figsize=fig_size, constrained_layout=True)
    ax_list = [axes] if cols == 1 else list(axes)

    ax_list[0].imshow(ov1)
    ax_list[0].set_title(f"Clean SAM Masks ({n1} masks)")
    ax_list[0].axis("off")

    if attacked_entry is not None:
        seg2 = attacked_entry.get("segmentation")
        ov2 = _render_mask_overlay(
            seg2, int(attacked_entry["height"]), int(attacked_entry["width"]), base_rgb=attacked_rgb, alpha=alpha
        )
        n2 = 0 if seg2 is None else len(seg2)
        ax_list[1].imshow(ov2)
        ax_list[1].set_title(f"Attacked SAM Masks ({n2} masks)")
        ax_list[1].axis("off")

    plt.show()
    return fig

In [ ]:
def visualize_sam_encoding(entry_or_image, grid_size: int = 16, tolerance: float = 0.02, mask_generator=None):
    """Visualize SAM grid features and neighbor sequences using coverless_steg helpers."""
    base_rgb = None
    if isinstance(entry_or_image, (str, Path)):
        bgr = cv2.imread(str(entry_or_image))
        if bgr is None:
            raise ValueError(f"Could not read image: {entry_or_image}")
        base_rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        if mask_generator is None:
            raise ValueError("mask_generator is required when passing a raw image path.")
        masks = mask_generator.generate(base_rgb)
        height, width = base_rgb.shape[:2]
        segmentations = np.stack([m["segmentation"] for m in masks]) if masks else None
    else:
        segmentations = entry_or_image.get("segmentation")
        height, width = int(entry_or_image["height"]), int(entry_or_image["width"])
        im_path = entry_or_image.get("image_path", "")
        if im_path and Path(im_path).is_file():
            bgr = cv2.imread(str(im_path))
            if bgr is not None:
                base_rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    mask_vis = _render_mask_overlay(segmentations, height, width, base_rgb=None)
    count_matrix, binary_matrix = compute_mask_matrices(segmentations, height, width)
    count_features = calculate_grid_means(count_matrix, grid_size=grid_size)
    binary_features = calculate_grid_means(binary_matrix, grid_size=grid_size)
    count_sequence = compare_neighbors(count_features, tolerance=tolerance)
    binary_sequence = compare_neighbors(binary_features, tolerance=tolerance)
    mask_count = 0 if segmentations is None else len(segmentations)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    axes[0].imshow(base_rgb if base_rgb is not None else mask_vis)
    axes[0].set_title(f"{'Original Image' if base_rgb is not None else 'SAM Mask Canvas'} ({width}x{height})")
    axes[1].imshow(mask_vis)
    axes[1].set_title(f"SAM Masks ({mask_count} masks)")
    axes[2].imshow(binary_matrix, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"Binary Coverage ({np.mean(binary_matrix) * 100:.1f}%)")
    for ax in axes:
        ax.axis("off")
    plt.show()

    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    for ax, (feat, title) in zip(
        axes2,
        [(count_features, "Mean Mask Count / Cell"), (binary_features, "Mean Binary Coverage / Cell")],
    ):
        im = ax.imshow(feat, cmap="viridis")
        ax.set_title(f"{title} ({grid_size}x{grid_size})")
        ax.set_xticks(np.arange(grid_size))
        ax.set_yticks(np.arange(grid_size))
        if grid_size <= 8:
            for r in range(grid_size):
                for c in range(grid_size):
                    val = feat[r, c]
                    ax.text(
                        c, r, f"{val:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if val < feat.max() * 0.65 else "black",
                    )
        fig2.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

    return {
        "count_matrix": count_matrix,
        "binary_matrix": binary_matrix,
        "count_features": count_features,
        "binary_features": binary_features,
        "count_sequence": count_sequence,
        "binary_sequence": binary_sequence,
        "image_width": width,
        "image_height": height,
        "mask_count": mask_count,
    }

In [ ]:
with h5py.File(CLEAN_H5, "r") as h5_clean:
    clean_entry = H5Entry(h5_clean["0000"], "0000")
    sample_canvas = _render_mask_overlay(
        clean_entry.get("segmentation"), clean_entry["height"], clean_entry["width"]
    )
    visualize_attack(
        sample_canvas,
        apply_gaussian_noise(sample_canvas, variance=0.005, rng=42),
        "In-Memory Gaussian Noise",
    )
    visualize_attack(
        sample_canvas,
        apply_jpeg_comp(sample_canvas, quality=50),
        "In-Memory JPEG (Q=50)",
    )

    if NOISE_H5.exists():
        with h5py.File(NOISE_H5, "r") as h5_noise:
            visualize_masks(clean_entry, H5Entry(h5_noise["0000"], "0000"))
    else:
        visualize_masks(clean_entry)

    encoding_info = visualize_sam_encoding(clean_entry, grid_size=4)
    print("4x4 count_sequence :", encoding_info["count_sequence"])
    print("4x4 binary_sequence:", encoding_info["binary_sequence"])